# Fuzzy Sets in Python

In this notebook, we will use **scikit-fuzzy** to build, visualize, and interpret fuzzy membership functions.

## Learning objectives
By the end of this notebook, you should be able to:

1. Explain what a **membership grade** means.
2. Create a trapezoidal membership function with `skfuzzy.fuzz.trapmf`.
3. Build a linguistic variable such as **Age** with terms like *Young*, *Middle*, and *Old*.
4. Read the membership grades of a crisp input from sampled fuzzy sets.
5. Connect the Python representation to the mathematical trapezoidal membership function.

> **Key idea:** A fuzzy membership grade is a value between 0 and 1. It is **not** a probability. A person can belong to more than one fuzzy set at the same time.

## 1. Setup

We will use:

- **NumPy** for the universe of discourse,
- **Matplotlib** for plotting, and
- **scikit-fuzzy** for membership functions.

If `scikit-fuzzy` is not installed in your environment, uncomment and run the installation line once.

Useful reference: <https://scikit-fuzzy.readthedocs.io/>

In [ ]:
# Uncomment once if needed:
# %pip install -U scikit-fuzzy

import numpy as np
import matplotlib.pyplot as plt
import skfuzzy as fuzz

%matplotlib inline

## 2. A first trapezoidal membership function

A trapezoidal membership function is commonly described by four parameters:

\[
[a,b,c,d]
\]

For a standard trapezoid:

- membership is 0 before \(a\),
- it rises from \(a\) to \(b\),
- it stays at 1 from \(b\) to \(c\),
- it falls from \(c\) to \(d\), and
- membership is 0 after \(d\).

The library evaluates the membership function over a **sampled universe of discourse** rather than storing only a symbolic formula. That means the spacing of `x` determines which crisp values are represented directly.

### Before you run the next cell
Predict the membership grade near \(x=2.25\), \(x=2.75\), and \(x=4.0\).

In [ ]:
# Universe of discourse: 0 to 5, sampled every 0.1
x = np.arange(0, 5.01, 0.1)

# Trapezoid parameters [a, b, c, d]
mfx = fuzz.trapmf(x, [2, 2.5, 3, 4.5])

plt.figure(figsize=(9, 5))
plt.plot(x, mfx, linewidth=2, label='Trapezoidal membership')
plt.fill_between(x, 0, mfx, alpha=0.25)
plt.xlabel('x')
plt.ylabel('Membership grade, μ(x)')
plt.title('Example trapezoidal membership function')
plt.ylim(-0.05, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

### Checkpoint

Answer these before continuing:

1. Which interval has full membership, \(\mu(x)=1\)?
2. Why is the function stored as an array of membership grades?
3. What would change if the sampling step were `0.01` instead of `0.1`?

## 3. Linguistic variable: Age

Now define the linguistic variable **Age** with three linguistic terms:

$\{Young,Middle,Old\}$

These sets are intentionally allowed to **overlap**. Overlap is a feature of fuzzy sets: a crisp value can have partial membership in several linguistic terms at once.

> The numerical boundaries below are examples for learning the mechanics. They are not claims about how age should be categorized in a real application.

In [ ]:
# Universe of discourse for age
age = np.arange(0, 101, 1)

# Fuzzy terms
young = fuzz.trapmf(age, [0, 0, 10, 20])
middle = fuzz.trapmf(age, [10, 20, 40, 50])
old = fuzz.trapmf(age, [30, 50, 100, 100])

plt.figure(figsize=(10, 5))
plt.plot(age, young, linewidth=2, label='Young')
plt.plot(age, middle, linewidth=2, label='Middle')
plt.plot(age, old, linewidth=2, label='Old')
plt.fill_between(age, 0, young, alpha=0.12)
plt.fill_between(age, 0, middle, alpha=0.12)
plt.fill_between(age, 0, old, alpha=0.12)
plt.xlabel('Age (years)')
plt.ylabel('Membership grade, μ(age)')
plt.title('Fuzzy linguistic variable: Age')
plt.ylim(-0.05, 1.05)
plt.xlim(0, 100)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 4. What are the membership grades for age 40?

Be careful with the wording: **“a 40-year-old”** means the crisp input \(x=40\). It does **not** mean “40 and older.”

Before running the code, inspect the graph and predict:

  * $\{\mu_{Young}(40), \mu_{Middle}(40), \mu_{Old}(40)\}$

In [ ]:
crisp_age = 40

# Because the universe is [0, 1, 2, ..., 100], the array index equals the age.
young_40 = young[crisp_age]
middle_40 = middle[crisp_age]
old_40 = old[crisp_age]

print(f'For age {crisp_age}:')
print(f'  Young  = {young_40:.2f}')
print(f'  Middle = {middle_40:.2f}')
print(f'  Old    = {old_40:.2f}')

plt.figure(figsize=(10, 5))
plt.plot(age, young, linewidth=2, label='Young')
plt.plot(age, middle, linewidth=2, label='Middle')
plt.plot(age, old, linewidth=2, label='Old')
plt.axvline(crisp_age, linestyle='--', linewidth=2, label=f'Age = {crisp_age}')
plt.scatter([crisp_age]*3, [young_40, middle_40, old_40], zorder=5)
plt.xlabel('Age (years)')
plt.ylabel('Membership grade, μ(age)')
plt.title('Membership grades for a crisp age of 40')
plt.ylim(-0.05, 1.05)
plt.xlim(0, 100)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

### Important indexing note

The expression `young[40]` works here only because the universe was created as

```python
age = np.arange(0, 101, 1)
```

so index `40` corresponds exactly to age 40.

If the universe used a different starting point or sampling interval, the array index would **not necessarily equal the crisp input**. In that case, interpolation is safer:

```python
fuzz.interp_membership(age, young, 40)
```

In [ ]:
# A more general way to evaluate a crisp input
crisp_age = 40

print('Using interpolation:')
print(f"  Young  = {fuzz.interp_membership(age, young, crisp_age):.2f}")
print(f"  Middle = {fuzz.interp_membership(age, middle, crisp_age):.2f}")
print(f"  Old    = {fuzz.interp_membership(age, old, crisp_age):.2f}")

## 5. Connecting the code to the mathematics

For trapezoidal parameters \((a,b,c,d)\), a compact mathematical form is

 * $t_{(a,b,c,d)}(x) = \max( \min( \frac{x-a}{b-a}, 1, \frac{d-x}{d-c} ), 0 )$

This compact form describes the rising edge, flat top, and falling edge of the trapezoid.

### Work it out by hand
For the set

* Middle = trapmf([10,20,40,50])

calculate the membership grade at:

- \(x=15\)
- \(x=30\)
- \(x=45\)

Then compare your results with Python below.

In [ ]:
for value in [15, 30, 45]:
    mu = fuzz.interp_membership(age, middle, value)
    print(f'μ_Middle({value}) = {mu:.2f}')

## 6. Explore other membership functions

Scikit-fuzzy includes several common membership-function shapes. For each one below:

1. Look up its parameters.
2. Sketch what you expect it to look like.
3. Plot it in Python.
4. Explain one situation where that shape might be useful.

Functions to explore:

- `trapmf` — trapezoidal
- `zmf` — Z-shaped
- `smf` — S-shaped
- `sigmf` — sigmoid
- `gaussmf` — Gaussian

Reference: <https://scikit-fuzzy.readthedocs.io/en/latest/api/skfuzzy.membership.html>

## 7. Student practice

### Exercise 1 — Modify a fuzzy term
Change the definition of `middle` so that full membership occurs from ages 30 through 45. Plot your new set together with `young` and `old`.

### Exercise 2 — Evaluate crisp ages
Find the memberships in `Young`, `Middle`, and `Old` for ages **18, 35, 55, and 75**. Put the results in a small table.

### Exercise 3 — Reason about overlap
Choose one age that belongs partially to **two** sets. Explain, in one or two sentences, why this does not create a contradiction in fuzzy logic.

### Exercise 4 — Sampling
Create a new universe sampled every `0.25` years. What changes in the arrays? What does not change conceptually?

### Challenge — Build your own linguistic variable
Choose a variable such as temperature, speed, risk, workload, or distance. Define at least three overlapping fuzzy terms and plot them. Briefly justify your chosen boundaries.

In [ ]:
# Workspace for Exercise 2
ages_to_test = [18, 35, 55, 75]

# TODO: compute and print the three membership grades for each age.

## 8. Takeaways

- A fuzzy set maps each input to a membership grade in \([0,1]\).
- Membership grades are **degrees of belonging**, not probabilities.
- Linguistic terms can overlap.
- Scikit-fuzzy evaluates membership functions over a sampled universe of discourse.
- `fuzz.interp_membership(...)` is a robust way to evaluate crisp values when the universe spacing does not line up exactly with array indices.
- The code and the mathematical membership-function definition describe the same underlying idea.